# AWS Bedrock + LiteLLM Testing

Simple tests for Bedrock models via LiteLLM with cost tracking.

In [4]:
# Imports
import os

import dspy
from datasets import load_dataset
from dotenv import load_dotenv
from dspy import GEPA, LM
from litellm import completion, completion_cost

load_dotenv()
print("✓ Loaded")

✓ Loaded


In [5]:
# Config
region = os.getenv("AWS_REGION")
token = os.getenv("AWS_BEARER_TOKEN_BEDROCK")
llama_1b = os.getenv("BEDROCK_LLAMA_1B")
llama_3b = os.getenv("BEDROCK_LLAMA_3B")
llama_8b = os.getenv("BEDROCK_LLAMA_8B")
gpt_oss_20b = os.getenv("BEDROCK_GPT_OSS_20B")
gpt_oss_120b = os.getenv("BEDROCK_GPT_OSS_120B")

print(f"Region: {region}")
print(f"Token: {'✓' if token else '✗'}")
print(f"Llama Models: {llama_1b}, {llama_3b}, {llama_8b}")
print(f"GPT OSS Models: {gpt_oss_20b}, {gpt_oss_120b}")

Region: us-east-1
Token: ✓
Llama Models: us.meta.llama3-2-1b-instruct-v1:0, us.meta.llama3-2-3b-instruct-v1:0, us.meta.llama3-1-8b-instruct-v1:0
GPT OSS Models: openai.gpt-oss-20b-1:0, openai.gpt-oss-120b-1:0


## Test 1: Basic Call

In [6]:
response = completion(
    model=f"bedrock/{llama_1b}",
    messages=[{"role": "user", "content": "Say hello in 5 words"}],
    max_tokens=50,
    temperature=0,
)

print(f"Response: {response.choices[0].message.content}")
print(f"Tokens: {response.usage.total_tokens}")
print(f"Cost: ${completion_cost(completion_response=response):.6f}")

Response: Hello, it's nice to meet you.
Tokens: 51
Cost: $0.000005


## Test 1b: GPT OSS 120B

In [7]:
response = completion(
    model=f"bedrock/{gpt_oss_120b}",
    messages=[{"role": "user", "content": "Say hello in 5 words"}],
    max_tokens=200,
    temperature=0,
)

print(f"Response: {response.choices[0].message.content}")
print(f"Tokens: {response.usage.total_tokens}")
print(f"Cost: ${completion_cost(completion_response=response):.6f}")

Response: Hello there, nice to meet.
Tokens: 211
Cost: $0.000095


## Test 2: Math Reasoning

In [ ]:
response = completion(
    model=f"bedrock/{llama_8b}",
    messages=[
        {"role": "system", "content": "Think step by step"},
        {
            "role": "user",
            "content": "If I have 3 apples and buy 2 more, then give 1 away, how many?",
        },
    ],
    max_tokens=150,
    temperature=0,
)

print(response.choices[0].message.content)
print(f"\nTokens: {response.usage.total_tokens}")
print(f"Cost: ${completion_cost(completion_response=response):.6f}")



Let's break it down step by step:

1. You start with 3 apples.
2. You buy 2 more apples, so now you have 3 + 2 = 5 apples.
3. You give 1 apple away, so now you have 5 - 1 = 4 apples.

So, you have 4 apples.

Tokens: 114
Cost: $0.000025


## Test 3: Compare All Models

In [9]:
question = "What is 2+2?"
total_cost = 0

for name, model in [("1B", llama_1b), ("3B", llama_3b), ("8B", llama_8b)]:
    response = completion(
        model=f"bedrock/{model}",
        messages=[{"role": "user", "content": question}],
        max_tokens=20,
        temperature=0,
    )

    cost = completion_cost(completion_response=response)
    total_cost += cost

    print(f"{name}: {response.choices[0].message.content}")
    print(f"     Tokens: {response.usage.total_tokens}, Cost: ${cost:.6f}\n")

print(f"Total: ${total_cost:.6f}")

1B: 2 + 2 = 4.
     Tokens: 51, Cost: $0.000005

3B: 2 + 2 = 4
     Tokens: 50, Cost: $0.000007

8B: 

2 + 2 = 4
     Tokens: 31, Cost: $0.000007

Total: $0.000019


## Test 5: Temperature Effects

In [10]:
prompt = "The future of AI is"

for temp in [0.0, 0.7, 1.0]:
    response = completion(
        model=f"bedrock/{llama_3b}",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=30,
        temperature=temp,
    )

    print(f"Temp {temp}: {response.choices[0].message.content}\n")

Temp 0.0: The future of AI is a topic of much debate and speculation. Here are some potential trends and developments that could shape the future of AI:

1.

Temp 0.7: The future of AI is vast and holds numerous possibilities. Here are some potential developments that could shape the future of AI:

1. **Increased Adoption in

Temp 1.0: The future of AI is likely to be shaped by several key trends and developments. Here are some potential directions:

1. **Increased adoption across industries**:



## Test 6: DSPy with LiteLLM

In [11]:
# Configure DSPy to use LiteLLM with Bedrock
lm = LM(model=f"bedrock/{llama_8b}", temperature=0, max_tokens=100)
dspy.configure(lm=lm)


# Define a simple signature
class BasicQA(dspy.Signature):
    """Answer questions concisely."""

    question: str = dspy.InputField()
    answer: str = dspy.OutputField()


# Create and test predictor
qa = dspy.ChainOfThought(BasicQA)
question = "What is the capital of France?"
response = qa(question=question)

print(f"Question: {question}")
print(f"Answer: {response.answer}")

# Inspect available fields
print(f"\nAvailable fields: {list(response.keys())}")

# Try to access reasoning if available
if hasattr(response, "reasoning"):
    print(f"\nReasoning: {response.reasoning}")
elif "reasoning" in response:
    print(f"\nReasoning: {response.reasoning}")

Question: What is the capital of France?
Answer: Paris

Available fields: ['reasoning', 'answer']

Reasoning: The capital of France is a well-known fact that can be easily looked up in a world atlas or a reliable online source. It is a widely accepted piece of information that is commonly taught in schools and is a basic fact that many people know.


In [12]:
# Access DSPy call history for cost tracking

# Get the last call from DSPy's history
last_call = lm.history[-1]

print("DSPy Call History:")
print(f"  Prompt tokens: {last_call['usage']['prompt_tokens']}")
print(f"  Completion tokens: {last_call['usage']['completion_tokens']}")
print(f"  Total tokens: {last_call['usage']['total_tokens']}")

# Calculate cost from the response
if "response" in last_call:
    cost = completion_cost(completion_response=last_call["response"])
    print(f"  Cost: ${cost:.6f}")
else:
    print("\n  Note: Raw response not available in history")
    print("  You can enable this with: lm = LM(..., cache=False)")

DSPy Call History:
  Prompt tokens: 169
  Completion tokens: 68
  Total tokens: 237
  Cost: $0.000052


# GEPA/DSPy HotpotQA Task

Testing prompt optimization on multi-hop question answering.

In [13]:
def init_dataset(num_samples=100):
    # Load HotpotQA dataset
    raw_data = load_dataset("hotpot_qa", "fullwiki")

    # Process train split
    raw_train = (
        raw_data["train"].shuffle(seed=42).select(range(min(len(raw_data["train"]), num_samples)))
    )
    train_split = [
        dspy.Example(
            {
                "question": x["question"],
                "answer": x["answer"],
            }
        ).with_inputs("question")
        for x in raw_train
    ]

    # Process validation split for test
    raw_val = (
        raw_data["validation"]
        .shuffle(seed=42)
        .select(range(min(len(raw_data["validation"]), num_samples)))
    )
    test_split = [
        dspy.Example(
            {
                "question": x["question"],
                "answer": x["answer"],
            }
        ).with_inputs("question")
        for x in raw_val
    ]

    # Split train into train/val
    tot_num = len(train_split)
    train_set = train_split[: int(0.5 * tot_num)]
    val_set = train_split[int(0.5 * tot_num) :]
    test_set = test_split

    return train_set, val_set, test_set


train_set, val_set, test_set = init_dataset(num_samples=100)
print(f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")
print(f"Example: {train_set[0]}")

README.md: 0.00B [00:00, ?B/s]

fullwiki/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

fullwiki/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

fullwiki/validation-00000-of-00001.parqu(…):   0%|          | 0.00/28.0M [00:00<?, ?B/s]

fullwiki/test-00000-of-00001.parquet:   0%|          | 0.00/27.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Train: 50, Val: 50, Test: 100
Example: Example({'question': 'Which airport is located in Maine, Sacramento International Airport or Knox County Regional Airport?', 'answer': 'Knox County Regional Airport'}) (input_keys={'question'})


In [14]:
# Configure DSPy to use LiteLLM with Bedrock (task agent)
lm = LM(model=f"bedrock/{gpt_oss_20b}", temperature=0.7, max_tokens=8192)
dspy.configure(lm=lm)

In [15]:
class GenerateResponse(dspy.Signature):
    """Answer the question with a short, factual response."""

    question = dspy.InputField()
    answer = dspy.OutputField(desc="A short factual answer (1-5 words)")


program = dspy.ChainOfThought(GenerateResponse)


def normalize_answer(s):
    """Normalize answer for comparison."""
    if s is None:
        return ""
    return s.lower().strip()


def metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    gold = normalize_answer(example["answer"])
    pred = normalize_answer(prediction.answer)
    if not pred:
        return 0
    # Exact match or gold contained in prediction
    return int(gold == pred or gold in pred or pred in gold)

In [16]:
evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    num_threads=32,
    display_table=True,
    display_progress=True,
)

evaluate(program)

Average Metric: 29.00 / 95 (30.5%):  95%|█████████▌| 95/100 [00:25<00:08,  1.62s/it]

2026/02/18 19:40:47 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=8192. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.7)  if the reason for truncation is repetition.


Average Metric: 29.00 / 96 (30.2%):  96%|█████████▌| 96/100 [00:30<00:09,  2.48s/it]

2026/02/18 19:40:50 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=8192. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.7)  if the reason for truncation is repetition.


Average Metric: 29.00 / 98 (29.6%):  98%|█████████▊| 98/100 [00:38<00:06,  3.32s/it]

2026/02/18 19:40:58 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=8192. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.7)  if the reason for truncation is repetition.


Average Metric: 29.00 / 100 (29.0%): 100%|██████████| 100/100 [01:06<00:00,  1.49it/s]

2026/02/18 19:41:23 INFO dspy.evaluate.evaluate: Average Metric: 29 / 100 (29.0%)


,question,example_answer,reasoning,pred_answer,metric
0,What nationality was Oliver Reed's character in the film Royal Flash?,Prussian,Oliver Reed played the aristocratic character Lord Bexley in the 1...,British,✔️ [0]
1,Pacific Mozart Ensemble performed which German composer's Der Lind...,Kurt Julian Weill,The Pacific Mozart Ensemble’s 2002 program included the contempora...,Günter Bialas,✔️ [0]
2,"Who released the song ""With or Without You"" first, Jai McDowall or...",U2,U2 released “With or Without You” in 1987 on the album *The Joshua...,U2,✔️ [1]
3,"What Kentucky county has a population of 60,316 and features the L...",Oldham County,"The 2010 U.S. Census lists the population of Floyd County, Kentuck...",Floyd County,✔️ [0]
4,"Para Hills West, South Australia lies within a city with what esti...","138,535","Para Hills West is a suburb of Adelaide, South Australia. The city...",≈ 1.4 million,✔️ [0]
...,...,...,...,...,...
95,"What company has a headquarters in Whitley, Convernty, United King...",Jaguar Land Rover,The village of Whitley in the county of Convernty (a fictional or ...,Whitley,✔️ [0]
96,Who starred as an American attorney in Grey Gardens?,Ken Howard,Grey Gardens is a 1975 documentary that includes a cameo by an Ame...,Anthony,✔️ [0]
97,Was Princess Charlotte of Cambridge born before or after the repea...,after,The Royal Marriages Act 1772 was repealed by the Succession to the...,After,✔️ [1]
98,Hermann Wilhelm Göring was a veteran fighter pilot in a war that e...,1918,"Hermann Göring served as a fighter pilot during World War I, which...",1918,✔️ [1]


EvaluationResult(score=29.0, results=<list of 100 results>)

In [17]:
def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    gold = normalize_answer(example["answer"])
    pred = normalize_answer(prediction.answer)

    # Handle None/empty answer
    if not pred:
        feedback_text = (
            "Your response was empty or could not be parsed. "
            "Please provide a short, factual answer to the question."
        )
        feedback_text += f" The correct answer is '{example['answer']}'."
        return dspy.Prediction(score=0, feedback=feedback_text)

    # Check for match
    score = int(gold == pred or gold in pred or pred in gold)

    if score == 1:
        feedback_text = f"Correct! The answer is '{example['answer']}'."
    else:
        feedback_text = f"Incorrect. You answered '{prediction.answer}', but the correct answer is '{example['answer']}'."
        feedback_text += " Make sure to provide a concise, factual answer."

    return dspy.Prediction(score=score, feedback=feedback_text)

In [18]:
optimizer = GEPA(
    metric=metric_with_feedback,
    auto="light",
    num_threads=64,
    track_stats=True,
    reflection_minibatch_size=3,
    # Using gpt_oss_120b for reflection - larger model for instruction generation
    reflection_lm=dspy.LM(model=f"bedrock/{gpt_oss_120b}", temperature=1.0, max_tokens=4096),
)

optimized_program = optimizer.compile(
    program,
    trainset=train_set,
    valset=val_set,
)

2026/02/18 19:41:23 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 580 metric calls of the program. This amounts to 5.80 full evals on the train+val set.
2026/02/18 19:41:23 INFO dspy.teleprompt.gepa.gepa: Using 50 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.
GEPA Optimization:   0%|          | 0/580 [00:00<?, ?rollouts/s]2026/02/18 19:41:53 INFO dspy.evaluate.evaluate: Average Metric: 19.0 / 50 (38.0%)
2026/02/18 19:41:53 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.38 over 50 / 50 examples
GEPA Optimization:   9%|▊         | 50/580 [00:29<05:16,  1.68rollouts/s]2026/02/18 19:41:53 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.38


Average Metric: 1.00 / 3 (33.3%): 100%|██████████| 3/3 [00:11<00:00,  3.75s/it] 

2026/02/18 19:42:04 INFO dspy.evaluate.evaluate: Average Metric: 1.0 / 3 (33.3%)


2026/02/18 19:42:09 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Proposed new text for predict: Task: Provide a single, concise, factual answer to each question.  
Do not include any reasoning, commentary, or extra text—only the answer itself on one line.

Guidelines  

1. **Exact Matching**  
   * The answer must match the expected response *exactly* in spelling, punctuation, capitalization, and formatting.  
   * Use plain ASCII characters only (e.g., straight apostrophe `'` rather than a typographic “curly” apostrophe).  

2. **Answer Format**  
   * If the question asks for a date, give the full date in the form **Month Day, Year** (e.g., `September 7, 1900`).  
   * If the question asks for a year only, give the four‑digit year (e.g., `1917`).  
   * If the question asks for a location, person name, occupation, title, etc., output the name exactly as it is conventionally written (including diacritics if they are part of the standard spelling, but still using plain ASCII for punctu

Average Metric: 1.00 / 3 (33.3%): 100%|██████████| 3/3 [00:07<00:00,  2.42s/it] 

...

2026/02/18 19:51:21 INFO dspy.evaluate.evaluate: Average Metric: 1.0 / 3 (33.3%)


2026/02/18 19:51:27 INFO dspy.teleprompt.gepa.gepa: Iteration 13: Proposed new text for predict: Task: Answer a trivia question with a brief, exact factual response.

Instructions:
1. Read the supplied question carefully.
2. Determine the single, precise answer that directly satisfies the question (e.g., a country, brand name, person’s name, etc.).
3. Respond **only** with that answer—no additional words, explanations, reasoning, or qualifiers.  
   - The answer should be as short as possible (typically 1–3 words) and correctly capitalized.
   - Do not include surrounding punctuation (quotes, parentheses, periods) unless the answer itself contains them.
4. If you are absolutely certain of the answer, output it exactly as it is commonly written (e.g., “Malta”, “Guinness”, “Winona Ryder”).  
5. If you cannot determine the answer with confidence, output the phrase: `I don't know.` (without quotes) and nothing else.
6. Do not attempt to guess or provide a partial answer; the correctness of

Average Metric: 2.00 / 3 (66.7%): 100%|██████████| 3/3 [00:17<00:00,  5.84s/it] 

2026/02/18 19:52:39 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 3 (66.7%)


2026/02/18 19:52:44 INFO dspy.teleprompt.gepa.gepa: Iteration 14: Proposed new text for predict: **Task Overview**
----------------
You will receive a single factual question. Your job is to return **only** the correct short factual answer—typically a name, title, term, or brief phrase—without any additional words, explanations, reasoning, or qualifiers.

**Core Guidelines**
-------------------
1. **Answer‑Only Output**  
   - Provide a single line containing **only** the answer (1‑3 words is typical).  
   - Do **not** prepend “Answer:”, add headings, bullet points, punctuation beyond what belongs to the answer itself, or any surrounding text.

2. **Confidence Requirement**  
   - If you are **confident** that the answer is correct, output it.  
   - If you cannot determine the answer with high confidence, output exactly:  
     ```
     I don't know
     ```

3. **No Hallucination**  
   - Do not fabricate information.  
   - Rely solely on verified factual knowledge.  
   - Do not i

In [ ]:
print(optimized_program.predict.signature.instructions)

You are to answer trivia‑style questions with a **single, concise, factual response**. Follow these rules exactly:

1. **Output only the answer** – no explanation, no headings, no surrounding text, no markdown.  
   - If the answer is a name, place, or occupation, output it exactly as the standard English term (e.g., `Astronomer`).  
   - If the answer is a date, output it in the form **Month Day, Year** (e.g., `September 7, 1900`).  
   - If the answer is a location comparison, output the name of the location that satisfies the query (e.g., `Woman's Viewpoint`).

2. **Answer must be factually correct**.  
   - Use reliable knowledge sources (internal knowledge base, vetted external data, or well‑known reference works).  
   - Do **not** guess or infer beyond the data you have; if you truly do not know, respond with `I don’t know`.

3. **Typical question types you will encounter**  
   - **Geographical comparison** – e.g., “Which is farther west, X or Y?” → compare longitudes; output t

In [ ]:
evaluate(optimized_program)

Average Metric: 29.00 / 98 (29.6%):  98%|█████████▊| 98/100 [00:45<00:06,  3.23s/it]

2026/02/11 12:19:49 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=8192. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.7)  if the reason for truncation is repetition.


Average Metric: 29.00 / 99 (29.3%):  99%|█████████▉| 99/100 [00:45<00:02,  2.43s/it]

2026/02/11 12:19:51 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=8192. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.7)  if the reason for truncation is repetition.


Average Metric: 29.00 / 100 (29.0%): 100%|██████████| 100/100 [00:48<00:00,  2.07it/s]

2026/02/11 12:19:51 INFO dspy.evaluate.evaluate: Average Metric: 29 / 100 (29.0%)


,question,example_answer,reasoning,pred_answer,metric
0,What nationality was Oliver Reed's character in the film Royal Flash?,Prussian,The character Oliver Reed played in *Royal Flash* was the prince o...,Rousonian,✔️ [0]
1,Pacific Mozart Ensemble performed which German composer's Der Lind...,Kurt Julian Weill,The piece “Der Lindberghflug” was composed by the German composer ...,Karlheinz Stockhausen,✔️ [0]
2,"Who released the song ""With or Without You"" first, Jai McDowall or...",U2,"U2 released ""With or Without You"" in 1987 on the album *The Joshua...",U2,✔️ [1]
3,"What Kentucky county has a population of 60,316 and features the L...",Oldham County,"The 2020 census lists Licking County, Kentucky with a population o...",Licking County,✔️ [0]
4,"Para Hills West, South Australia lies within a city with what esti...","138,535","Para Hills West is a suburb of Adelaide, South Australia. The esti...","1,359,000",✔️ [0]
...,...,...,...,...,...
95,"What company has a headquarters in Whitley, Convernty, United King...",Jaguar Land Rover,I don’t know,I don’t know,✔️ [0]
96,Who starred as an American attorney in Grey Gardens?,Ken Howard,The American attorney character in the 2009 film adaptation of *Gr...,John Glover,✔️ [0]
97,Was Princess Charlotte of Cambridge born before or after the repea...,after,The Royal Marriages Act 1772 was repealed by the Succession to the...,after,✔️ [1]
98,Hermann Wilhelm Göring was a veteran fighter pilot in a war that e...,1918,Hermann Wilhelm Göring served as a fighter pilot during World War ...,1918,✔️ [1]


EvaluationResult(score=29.0, results=<list of 100 results>)

Generative AI was used to assist in building this notebook.